# Django Authentication System

## Using Django's Built-in Auth

Instead of managing sessions manually, use Django's `authenticate()`, `login()`, and `logout()` functions.

```python
from django.contrib.auth import authenticate, login
from django.shortcuts import render, redirect
from django.http import HttpResponse

def login_view(request):
    if request.user.is_authenticated:
        return HttpResponse(f"{request.user.username} is already logged in.")

    if request.method == "POST":
        username = request.POST.get("username")
        password = request.POST.get("password")
        user = authenticate(request, username=username, password=password)
        if user:
            login(request, user)
            return redirect("profile")
        return render(request, "login.html", {"error": "Invalid credentials"})
    return render(request, "login.html")
```


## Logout and request.user

```python
from django.contrib.auth import logout
from django.contrib.auth.decorators import login_required

def logout_view(request):
    logout(request)
    return redirect("login")

@login_required
def profile_view(request):
    return HttpResponse(f"Welcome {request.user.username} - {request.user.email}")
```

### request.user attributes
- `request.user.is_authenticated` — True if logged in
- `request.user.username` — the username string
- `request.user.email` — the email address

### login_required setup
Add this to `settings.py` so unauthenticated users are redirected correctly:
```python
LOGIN_URL = '/login/'
```


## Sign-Up View

Create a new user with `User.objects.create_user()`:

```python
from django.contrib.auth.models import User

def signup_view(request):
    if request.method == "POST":
        username = request.POST.get("username")
        email = request.POST.get("email")
        pw1 = request.POST.get("password1")
        pw2 = request.POST.get("password2")

        if pw1 != pw2:
            return render(request, "signup.html", {"error": "Passwords do not match"})
        if User.objects.filter(username=username).exists():
            return render(request, "signup.html", {"error": "Username already taken"})

        user = User.objects.create_user(username=username, email=email, password=pw1)
        login(request, user)
        return redirect("profile")
    return render(request, "signup.html")
```

`create_user()` automatically hashes the password.


## Permissions

### Default Permissions
Django creates four permissions per model automatically: `add`, `change`, `delete`, `view`.

### Custom Permissions
Define them in the model's `Meta` class:
```python
class Review(models.Model):
    content = models.TextField()

    class Meta:
        permissions = [
            ("can_add_review", "Can add reviews"),
        ]
```

### Protecting Views with Permissions
```python
from django.contrib.auth.decorators import permission_required

@permission_required('myapp.can_add_review', raise_exception=True)
def add_review_view(request):
    return HttpResponse("Add review page")
```

Assign permissions to users through the Django admin.


## Custom Authentication Backend

A custom backend lets users log in with their email address instead of username:

```python
# accounts/backends.py
from django.contrib.auth.models import User
from django.contrib.auth.backends import ModelBackend

class EmailBackend(ModelBackend):
    def authenticate(self, request, username=None, password=None, **kwargs):
        try:
            user = User.objects.get(email=username)
            if user.check_password(password):
                return user
        except User.DoesNotExist:
            return None
```

Register in `settings.py`:
```python
AUTHENTICATION_BACKENDS = [
    'accounts.backends.EmailBackend',
    'django.contrib.auth.backends.ModelBackend',  # fallback
]
```


## Password Reset via Email

Django provides built-in password reset views. For development, print emails to the console:

```python
# settings.py
EMAIL_BACKEND = 'django.core.mail.backends.console.EmailBackend'
```

Add the built-in URLs:
```python
from django.contrib.auth import views as auth_views

urlpatterns += [
    path('reset/', auth_views.PasswordResetView.as_view(), name='reset'),
    path('reset_done/', auth_views.PasswordResetDoneView.as_view(), name='password_reset_done'),
    path('reset_confirm/<uidb64>/<token>/', auth_views.PasswordResetConfirmView.as_view(), name='password_reset_confirm'),
    path('reset_complete/', auth_views.PasswordResetCompleteView.as_view(), name='password_reset_complete'),
]
```


## Summary

- `authenticate()` validates credentials; `login()` starts the session; `logout()` ends it.
- `request.user` gives access to the logged-in user's attributes.
- `@login_required` redirects unauthenticated users to `LOGIN_URL`.
- `create_user()` creates a user with a properly hashed password.
- Custom permissions are defined in `Meta.permissions` and checked with `@permission_required`.
- Custom auth backends allow alternative login fields such as email.
- Use the console email backend to test password reset flows in development.
